<a href="https://colab.research.google.com/github/Amrishpurigoswami/Machine-Learning-Intern/blob/midnight/Machine%20Learning%20Intern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install langchain openai

In [70]:
import os
os.makedirs("data", exist_ok=True)

In [71]:
%%writefile data/knowledge.json
{
  "product": "AutoStream",
  "description": "AI-powered automated video editing platform for creators",

  "plans": {
    "basic": {
      "price": "$29/month",
      "features": ["10 videos/month", "720p resolution"],
      "best_for": ["beginners", "casual creators"]
    },
    "pro": {
      "price": "$79/month",
      "features": ["Unlimited videos", "4K resolution", "AI captions"],
      "best_for": ["YouTubers", "professional creators"]
    }
  },

  "platform_support": {
    "YouTube": {
      "use_case": "Long-form videos, monetization, ads",
      "recommended_plan": "Pro"
    },
    "Instagram": {
      "use_case": "Short reels and social content",
      "recommended_plan": "Basic"
    },
    "Spotify": {
      "use_case": "Podcasts and audio content",
      "recommended_plan": "Pro"
    },
    "Gaana": {
      "use_case": "Music distribution",
      "recommended_plan": "Pro"
    }
  },

  "ai_tools_context": {
    "ChatGPT": "Used for scripting and content generation",
    "Gemini": "Used for research and idea generation",
    "Claude": "Used for long-form writing"
  },

  "policies": {
    "refund": "No refunds after 7 days",
    "support": "24/7 support only on Pro plan"
  }
}

Overwriting data/knowledge.json


In [72]:
%%writefile intent.py
def detect_intent(user_input):
    text = user_input.lower()

    if any(word in text for word in ["hi", "hello", "hey"]):
        return "greeting"

    if any(word in text for word in ["want", "buy", "subscribe", "sign up", "interested"]):
        return "high_intent"

    if any(word in text for word in ["price", "pricing", "cost", "plan", "features"]):
        return "query"

    return "general"


Overwriting intent.py


In [73]:
%%writefile rag.py
import json

def load_knowledge():
    with open("data/knowledge.json", "r") as f:
        return json.load(f)

def get_answer(query, data):
    query = query.lower()

    if "price" in query or "plan" in query or "pricing" in query:
        return (
            f"Basic Plan: {data['plans']['basic']['price']}\n"
            f"Pro Plan: {data['plans']['pro']['price']}\n"
            "Pro is best for serious creators."
        )

    for platform in data["platform_support"]:
        if platform.lower() in query:
            info = data["platform_support"][platform]
            return (
                f"For {platform}: {info['use_case']}\n"
                f"Recommended Plan: {info['recommended_plan']}"
            )

    if "refund" in query:
        return data["policies"]["refund"]

    if "support" in query:
        return data["policies"]["support"]

    return "I can help with pricing, platforms, and plans."

Overwriting rag.py


In [74]:
%%writefile memory.py
class Memory:
    def __init__(self):
        self.history = []
        self.lead_data = {"name": None, "email": None, "platform": None}
        self.stage = "start"

    def update_history(self, user, bot):
        self.history.append({"user": user, "bot": bot})

Overwriting memory.py


In [75]:
%%writefile tools.py
def mock_lead_capture(name, email, platform):
    print(f"Lead captured successfully: {name}, {email}, {platform}")

Overwriting tools.py


In [78]:
%%writefile app.py
from rag import load_knowledge, get_answer
from memory import Memory
from tools import mock_lead_capture

memory = Memory()
data = load_knowledge()

def agent(user_input):
    text = user_input.lower()

    if memory.stage == "done":
        return "We’ve already captured your details. Our team will reach out soon! 🚀"

    if any(word in text for word in ["hi", "hello", "hey"]):
        return "Hi! Welcome to AutoStream 🚀 How can I help you?"

    if any(word in text for word in ["want", "buy", "subscribe", "sign up", "interested"]):
        if memory.stage == "start":
            memory.stage = "collect_name"
            return "Great! Let's get you started. What's your name?"

    if memory.stage == "collect_name":
        memory.lead_data["name"] = user_input
        memory.stage = "collect_email"
        return "Please provide your email."

    if memory.stage == "collect_email":
        memory.lead_data["email"] = user_input
        memory.stage = "collect_platform"
        return "Which platform do you create content on?"

    if memory.stage == "collect_platform":
        memory.lead_data["platform"] = user_input

        mock_lead_capture(
            memory.lead_data["name"],
            memory.lead_data["email"],
            memory.lead_data["platform"]
        )

        memory.stage = "done"
        return "You are all set! 🎉 Our team will contact you soon."

    if any(word in text for word in ["price", "pricing", "cost", "plan", "features"]):
        if memory.stage == "start":
            return get_answer(user_input, data)

    if memory.stage != "start":
        return "Let's complete your signup first 😊"

    return "Can you clarify your question?"

Overwriting app.py


In [79]:
import os
print(os.listdir())
print(os.listdir("data"))

['.config', '__pycache__', 'memory.py', 'app.py', 'tools.py', 'rag.py', 'data', 'intent.py', 'sample_data']
['knowledge.json']


In [80]:
import importlib
import app

importlib.reload(app)

<module 'app' from '/content/app.py'>

In [81]:
from app import agent, memory

memory.stage = "start"

print("AutoStream AI Agent Started (type 'exit' to stop)\n")

while True:
    user = input("You: ")

    if user.lower() == "exit":
        break

    response = agent(user)
    print("Bot:", response)

AutoStream AI Agent Started (type 'exit' to stop)

You: Hi
Bot: Hi! Welcome to AutoStream 🚀 How can I help you?
You: Tell me plan
Bot: Basic Plan: $29/month
Pro Plan: $79/month
Pro is best for serious creators.
You: I am interested in Pro plan for YouTube
Bot: Great! Let's get you started. What's your name?
You: Amrish
Bot: Please provide your email.
You: amrish@gmail.com
Bot: Which platform do you create content on?
You: YouTube
Lead captured successfully: Amrish, amrish@gmail.com, YouTube
Bot: You are all set! 🎉 Our team will contact you soon.
You: exit
